In [ ]:
import numpy as np
import matplotlib.pyplot as plt 
import qiskit as qsk 
from qiskit_aer import AerSimulator
import copy
from collections import deque
from own_types import *

# BB84 class is a util class created to store all the methods needed for the Protocol
# The class should 
# 
# 

class Upper_Block:
        i: int = 0

        def __init__(self, indeces: Bits, iter: int):
            self.id = f'BLK_{Upper_Block.__i}'
            Upper_Block.__i += 1
            self.indeces = indeces
            self.iter = iter 
            self.parity = -1

        @classmethod
        def reset(cls):
            cls.__i = 0
        
        def __str__(self) -> str:
            return f'Block: {self.id} | Indeces: {self.indeces} | Iter: {self.iter} | Parity: {self.parity}'



class BB84: 
    
    def __init__(self, no_bits: int, *, picking_rate: float = .7):
        self.a_bits: Bits = np.random.randint(0, 2, size=no_bits)
        self.a_bases: Bases = np.random.randint(0, 2, size=no_bits)
        self.b_bases: Bits = np.random.randint(0, 2, size=no_bits)
        self.circuits: Circs = []
        self.PICKING_RATE = picking_rate

    def __str__(self):
        return f'################### BB84 ###################\n Bits sent: {self.a_bits}\n Bases for encryption: {self.a_bases}\n\
        ###################\n Bases for decryption: {self.b_bases} \n PICKING RATE: {self.PICKING_RATE}'

    def __encodingPersending(self) -> None:
        if len(self.circuits) != 0:
            self.circuits.clear()

        if len(self.a_bases) != len(self.a_bits):
            raise Exception("####### Error in the encode_persending function <:> Length of the bases list should be equal to the length of bits list")

        for base, bit in zip(self.a_bases, self.a_bits):
            circ = QuantumCircuit(1, 1)
            # Z-base (vertical - |0> & |1>)
            if base == 0:
                # Apply a PauliX Gate on the qbit
                if bit == 1:
                    circ.x(0)
            # X-base (horizontal - |+> & |->)
            else:
                if bit == 1:
                    circ.x(0)
                circ.h(0)
            # Add the circuit to the list of circuits to be run after by the simulator
            self.circuits.append(circ)

    # For the measurement we simple choose a seq of random bases and whenever the base |+> & |-> we apply a hadamard gate to invert the 
    # qbit in the normal Z-base for |0> & |1> we leave the qbits as they are 
    def __measurements(self) -> None:
        if len(self.circuits) != len(self.b_bases):
            raise Exception("####### Error in the measure function <:> Length of the bases list should be equal to the length of circuits list")

        for circ, base in zip(self.circuits, self.b_bases):
            if base == 1:
                circ.h(0)
            circ.measure(0, 0)


    def __eavesdropping(self) -> list[tuple[Iter, Bit, Base]]:
        size = len(self.circuits)
        eve_choice = np.random.choice([True, False], p=[self.PICKING_RATE, 1-self.PICKING_RATE], size=size)

        # measure, write down and resend the qbit
        results: list[tuple[Iter, Bit, Base]] = []
        for i, qc in enumerate(self.circuits):
            # deciding eihter to measure or not
            if eve_choice[i]:
                # choose a bases for measurement at random
                base = np.random.randint(0, 2)
                if base == 1:
                    qc.h(0)
                qc.measure(0, 0)

                # Simulate the circuit and get the result 
                simulator = AerSimulator()
                trans = qsk.transpile(qc, simulator)
                job = simulator.run(trans, shots=1, memory=True)
                rez = job.result().get_memory(trans)
                results.append((i, rez[0], base))
                # re-prepering the qbit 
                qc_new = QuantumCircuit(1, 1)
                if base == 0:
                    if rez[0] == 1:
                        qc_new.x(0)
                else:
                    if rez[0] == 1:
                        qc_new.x(0)
                    qc_new.h(0)

                # replace the old circuite with the new one 
                qc = qc_new 

        return results

    
    # public methods
    # de implementat mai pe seara cand ajung acasa
    def run(self):
        pass
    




class CascadeReconcill:

    def __init__(self, protocol: BB84):
        self.b_bits = protocol.run()

    def binarySplit(self, a_bits: Bits, b_bits: Bits, indeces: list[int]) -> int:
        if len(indeces) == 1:
            return indeces[0]
        # recursive break the bit_block into two parts and select the one who presents an odd-error
        break_point = len(indeces) // 2
        left_parity = self.getParity(a_bits, b_bits, indeces[:break_point])
        if left_parity % 2 == 1:
            return self.binarySplit(a_bits, b_bits, indeces[:break_point])
        else: 
            return self.binarySplit(a_bits, b_bits, indeces[break_point:])
        

    def getParity(self, a_bits: Bits, b_bits: Bits, indeces: list[int]) -> int:
        count = 0
        for i in indeces:
            if self.a_bits[i] != b_bits[i]:
                count += 1
        return count 


In [ ]:
b = BB84()